# 🧪 Lab 01: Function Calling e Loop de Invocação de Ferramentas
**Módulo:** `04-ai-agents / function-calling-tools`

## Objetivos
1. Definir ferramentas Python com esquemas tipados (Tool Schemas).
2. Simular a decisão do LLM em invocar uma ferramenta específica com argumentos em JSON.
3. Executar dinamicamente a função correspondente e retornar o resultado para o contexto do agente.

In [1]:
import json
import inspect
from typing import Dict, Any, Callable

# 1. Definição das funções reais que o Agente pode invocar
def consultar_status_servico(nome_servico: str) -> dict:
    """Consulta o status operacional de um serviço de infraestrutura."""
    base_status = {
        "orionprov": {"status": "ONLINE", "latencia_ms": 12, "uptime": "99.98%"},
        "cadastra-ai": {"status": "DEGRADED", "latencia_ms": 340, "uptime": "98.50%"},
        "database-pg": {"status": "ONLINE", "latencia_ms": 2, "uptime": "100.00%"}
    }
    key = nome_servico.lower()
    if key in base_status:
        return {"servico": nome_servico, **base_status[key]}
    return {"servico": nome_servico, "status": "NOT_FOUND", "erro": "Serviço não catalogado"}

def testar_conectividade_ip(ip: str, porta: int = 80) -> dict:
    """Simula um teste de socket/ping para um endereço IP e porta específicos."""
    return {"ip": ip, "porta": porta, "alcancaveis": True, "rtt_ms": 18.5}

# Registro de ferramentas disponíveis no Tool Registry
TOOL_REGISTRY: Dict[str, Callable] = {
    "consultar_status_servico": consultar_status_servico,
    "testar_conectividade_ip": testar_conectividade_ip
}

## 2. Declaração do Schema da Ferramenta (Formato JSON Schema)

In [2]:
# Esquemas que seriam enviados no payload da chamada à API do LLM
tools_schema = [
    {
        "name": "consultar_status_servico",
        "description": "Retorna o status de saúde e métricas de um serviço técnico pelo nome.",
        "parameters": {
            "type": "object",
            "properties": {
                "nome_servico": {
                    "type": "string",
                    "description": "Nome do serviço a ser verificado (ex: orionprov, cadastra-ai, database-pg)"
                }
            },
            "required": ["nome_servico"]
        }
    },
    {
        "name": "testar_conectividade_ip",
        "description": "Testa se um IP e porta estão respondendo.",
        "parameters": {
            "type": "object",
            "properties": {
                "ip": {"type": "string", "description": "Endereço IPv4 de destino"},
                "porta": {"type": "integer", "description": "Porta TCP/UDP (padrão: 80)"}
            },
            "required": ["ip"]
        }
    }
]

print("✅ Schemas de ferramentas prontos para envio ao LLM:")
print(json.dumps(tools_schema, indent=2, ensure_ascii=False))

✅ Schemas de ferramentas prontos para envio ao LLM:
[
  {
    "name": "consultar_status_servico",
    "description": "Retorna o status de saúde e métricas de um serviço técnico pelo nome.",
    "parameters": {
      "type": "object",
      "properties": {
        "nome_servico": {
          "type": "string",
          "description": "Nome do serviço a ser verificado (ex: orionprov, cadastra-ai, database-pg)"
        }
      },
      "required": [
        "nome_servico"
      ]
    }
  },
  {
    "name": "testar_conectividade_ip",
    "description": "Testa se um IP e porta estão respondendo.",
    "parameters": {
      "type": "object",
      "properties": {
        "ip": {
          "type": "string",
          "description": "Endereço IPv4 de destino"
        },
        "porta": {
          "type": "integer",
          "description": "Porta TCP/UDP (padrão: 80)"
        }
      },
      "required": [
        "ip"
      ]
    }
  }
]


## 3. Loop de Execução e Invocação Dinâmica (Tool Execution Loop)

In [3]:
def agent_executor_loop(llm_tool_call_response: dict):
    """
    Recebe a decisão do LLM de invocar uma ferramenta, resolve a função
    no registro e executa com os argumentos parseados.
    """
    function_name = llm_tool_call_response.get("function")
    arguments = llm_tool_call_response.get("arguments", {})
    
    print(f"🤖 LLM decidiu invocar a ferramenta: '{function_name}'")
    print(f"⚙️ Argumentos fornecidos: {arguments}")
    
    if function_name in TOOL_REGISTRY:
        target_function = TOOL_REGISTRY[function_name]
        try:
            resultado = target_function(**arguments)
            print("⚡ Execução bem-sucedida! Resultado retornado:")
            print(json.dumps(resultado, indent=2, ensure_ascii=False))
            return resultado
        except Exception as e:
            print(f"❌ Erro ao executar a função: {e}")
            return {"erro": str(e)}
    else:
        print(f"❌ Ferramenta '{function_name}' não encontrada no registro.")
        return {"erro": "Ferramenta desconhecida"}

# Simulação 1: LLM solicitou checagem do OrionProv
chamada_simulada_1 = {
    "function": "consultar_status_servico",
    "arguments": {"nome_servico": "orionprov"}
}

# Simulação 2: LLM solicitou teste de conectividade IP
chamada_simulada_2 = {
    "function": "testar_conectividade_ip",
    "arguments": {"ip": "10.0.0.1", "porta": 443}
}

print("--- EXECUTANDO SIMULAÇÃO 1 ---")
agent_executor_loop(chamada_simulada_1)

print("\n--- EXECUTANDO SIMULAÇÃO 2 ---")
agent_executor_loop(chamada_simulada_2)

--- EXECUTANDO SIMULAÇÃO 1 ---
🤖 LLM decidiu invocar a ferramenta: 'consultar_status_servico'
⚙️ Argumentos fornecidos: {'nome_servico': 'orionprov'}
⚡ Execução bem-sucedida! Resultado retornado:
{
  "servico": "orionprov",
  "status": "ONLINE",
  "latencia_ms": 12,
  "uptime": "99.98%"
}

--- EXECUTANDO SIMULAÇÃO 2 ---
🤖 LLM decidiu invocar a ferramenta: 'testar_conectividade_ip'
⚙️ Argumentos fornecidos: {'ip': '10.0.0.1', 'porta': 443}
⚡ Execução bem-sucedida! Resultado retornado:
{
  "ip": "10.0.0.1",
  "porta": 443,
  "alcancaveis": true,
  "rtt_ms": 18.5
}


{'ip': '10.0.0.1', 'porta': 443, 'alcancaveis': True, 'rtt_ms': 18.5}